# 🇳🇵🇮🇳🇺🇸 Chatterbox Multilingual Fine-tuning

This notebook fine-tunes the Chatterbox TTS model on a combined **Nepali, Maithili, and English** dataset (27GB) using Kaggle T4 x2 GPUs.

**Note**: This version uses **Streaming Mode** to bypass Kaggle's 20GB disk limit. No audio extraction is required.

In [ ]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

# 1. Setup Environment
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# 2. Clone Repository
REPO_URL = "https://github.com/Firojpaudel/chatterbox-nepali.git"
REPO_DIR = "/kaggle/working/chatterbox-nepali"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Updating repository...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)

# 3. Install Dependencies
print("Installing dependencies...")
subprocess.run(["pip", "uninstall", "-y", "torchcodec"], check=False)
subprocess.run(["pip", "install", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run(["pip", "install", "-e", "."], check=True)
subprocess.run(["pip", "install", "pandas>=2.0,<2.3", "--force-reinstall", "--no-deps"], check=True)
subprocess.run(["pip", "install", "datasets", "soundfile", "omegaconf", "conformer", "s3tokenizer", "safetensors", "pyloudnorm", "wandb", "huggingface-hub", "librosa"], check=True)
subprocess.run(["pip", "install", "git+https://github.com/resemble-ai/Perth.git@master"], check=True)

print("\n--- Chatterbox Setup Complete. PLEASE RESTART SESSION NOW ---")

In [ ]:
%%writefile /kaggle/working/chatterbox-nepali/prepare_data.py
import os
from huggingface_hub import HfApi

# Verification script
api = HfApi()
try:
    user = api.whoami(token=os.environ.get("HF_TOKEN"))
    print(f"✅ Logged in as {user['name']}")
    print("🚀 Streaming Mode enabled: Dataset will be loaded directly from HF during training.")
except Exception as e:
    print(f"❌ HF Login failed: {e}")

In [ ]:
!python /kaggle/working/chatterbox-nepali/prepare_data.py

In [ ]:
# 🚀 START MULTILINGUAL TRAINING (T4 x 2)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  torchrun --nproc_per_node=2 src/chatterbox/train_multilingual.py \
    --repo_id Firoj112/chatterbox-multilingual-data \
    --push_to_hub Firoj112/chatterbox-multilingual-t3-v1 \
    --batch_size 8 \
    --accum_steps 2 \
    --epochs 50 \
    --lr 5e-5 \
    --num_workers 1 \
    --distributed \
    --fp16